# Tuning the threshold while training the model

Code repository for the book:

[Imbalanced Data: Myths, Mistakes and Modern Solutions](https://www.trainindata.com/p/imbalanced-data-myths-mistakes-solutions-book)

In this notebook, we'll optimise the threshold while training the machine learning model.

Scikit-learn's `TunedThresholdClassifierCV` fits the estimator on each cross-validation fold and computes the chosen metric across a grid of candidate thresholds on the corresponding held-out fold. By default, it examines 100 cut-off points equidistantly separated.

It then averages these score curves across folds and picks the single threshold that maximises the averaged curve. 

In [1]:
import numpy as np
from imblearn.datasets import fetch_datasets
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score

In [2]:
data = fetch_datasets()["wine_quality"]
y = np.where(data.target < 0, 0, 1)

X_train, X_test, y_train, y_test = train_test_split(
    data.data, y, test_size=0.3, random_state=0, stratify=y)

In [3]:
IR = 1 / y_train.mean()

# Elkan's threshold when we use IR as class weight
theoretical_thres = 1 / (1+IR)

theoretical_thres

np.float64(0.03599550056242969)

## Baseline: default 0.5 threshold

In [4]:
gbm = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
preds = gbm.predict(X_test)
balanced_accuracy_score(y_test, preds)

0.5736909733376164

## Training the Model and Tuning the Decision Threshold

In [5]:
tuned = TunedThresholdClassifierCV(
    GradientBoostingClassifier(random_state=0),
    scoring="balanced_accuracy",
    cv=5,
)
tuned.fit(X_train, y_train)
tuned.best_threshold_

np.float64(0.022202891294731332)

In [6]:
tuned_preds = tuned.predict(X_test)
balanced_accuracy_score(y_test, tuned_preds)

0.8027304850626406

The tuned threshold moves well away from 0.5 and recovers a much higher balanced accuracy than the default cut-off.

## Doing it manually: threshold dispersion across folds

`TunedThresholdClassifierCV` averages the score curves across folds before picking a single threshold, so it hides how much that threshold varies from fold to fold. To see that dispersion, we can pick the best threshold on each fold ourselves and look at the spread.

In [7]:
from sklearn.model_selection import StratifiedKFold

thresholds = np.linspace(0.01, 0.99, 50)

def best_threshold(y_true, probs):
    scores = [balanced_accuracy_score(y_true, probs >= t) for t in thresholds]
    return thresholds[np.argmax(scores)]

In [8]:
cv_thresholds, cv_scores = [], []

for train_idx, val_idx in StratifiedKFold(5).split(X_train, y_train):
    gbm_cv = GradientBoostingClassifier(random_state=0).fit(X_train[train_idx], y_train[train_idx])
    probs = gbm_cv.predict_proba(X_train[val_idx])[:, 1]
    t = best_threshold(y_train[val_idx], probs)
    cv_thresholds.append(t)
    cv_scores.append(balanced_accuracy_score(y_train[val_idx], probs >= t))

In [9]:
print(f"Threshold: {np.mean(cv_thresholds):.3f} +- {np.std(cv_thresholds):.3f}")
print(f"Balanced accuracy: {np.mean(cv_scores):.3f} +- {np.std(cv_scores):.3f}")

Threshold: 0.026 +- 0.008
Balanced accuracy: 0.773 +- 0.032
